In [ ]:
import pandas
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

data = pandas.read_csv("reviews.csv")

**Zad?n?:** V tomto cvi?en? zkus naimplementovat klasifik?tor, kter? rozpozn? pozitivn? recenzi produktu od negativn?. Pracuj se souborem [reviews.csv](reviews.csv). Soubor obsahuje text recenze, a v?stupn? prom?nnou `label` se dv?ma hodnotami, `neg` a `pos`. M??e? si v?imnout, ?e data u? n?jak?m ?i?t?n?m pro?la. Porovnej alespo? dva algoritmy. Jak? je nejlep?? f1-score, kter?ho jsi dos?hl/a?

V tomto cvi?en? zkus naimplementovat klasifik?tor, kter? rozpozn? pozitivn? recenzi produktu od negativn?. Pracuj se souborem [reviews.csv](reviews.csv). Soubor obsahuje text recenze, a v?stupn? prom?nnou `label` se dv?ma hodnotami, `neg` a `pos`. M??e? si v?imnout, ?e data u? n?jak?m ?i?t?n?m pro?la. Porovnej alespo? dva algoritmy. Jak? je nejlep?? f1-score, kter?ho jsi dos?hl/a?

In [ ]:
X = data["review"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
X_train_text = X_train
X_train_text = X_train_text.reset_index(drop=True)
# Pou?ijeme TfidfVectorizer s anglick?mi stop words, samostatn? slova a dvojice slov
vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
X_train = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

# Jako algoritmus vyu?ijeme Support Vector Machine
clf = LinearSVC()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test_vec)
print(accuracy_score(y_test, y_pred))
print(f1_score(y_test, y_pred, pos_label="pos"))

V?sledn? model m? accuracy 82.1 % a f1-score 81.8 %

M??eme si vyzkou?et funkci `classification_report`, kter? vyp??e v?echny metriky jako tabulku.

In [ ]:
report_2 = classification_report(y_test, y_pred)
print(report_2)

Vedle SVM m??eme vyzkou?et i algoritmus K Nearest Neighbors, kter? dosahuje ni??? accuracy i f1-score (oboj? kolem 65 %).

In [ ]:
clf_2 = KNeighborsClassifier()
clf_2.fit(X_train, y_train)
y_pred_knn = clf_2.predict(X_test_vec)
print(accuracy_score(y_test, y_pred_knn))
print(f1_score(y_test, y_pred_knn, pos_label="pos"))

**Bonus 1:** Identifikuj ty recenze, kter? tv?j model ozna?il ?patn?. Na p?r z nich se pod?vej, a zkus odvodit, pro? model chyboval.

Vytvo??me si nyn? pandas tabulku `df_vysledek`, do kter? si ulo??me text recenze a skute?nou a predikovanou hodnotu label. M??eme pou??t dva zp?soby: Prvn? z nich je vytvo?en? cel? tabulky najednou, a to s vyu?it?m slovn?ku, do kter?ho vlo??me jednotliv? sloupce.

In [ ]:
df_vysledek = pandas.DataFrame({"text": X_test, "y_test": y_test, "y_pred": y_pred}).reset_index(drop=True)
df_vysledek

Nyn? si vytvo??me tabulku, do kter? vlo??me pouze ty ??dky, kter? se predikovan? a skute?n? label li??.

In [ ]:
df_vysledek_incorrect = df_vysledek[df_vysledek["y_test"] != df_vysledek["y_pred"]]
df_vysledek_incorrect.to_csv("errors.csv", index=False)
df_vysledek_incorrect = df_vysledek_incorrect.reset_index()
df_vysledek_incorrect.head()

Druh? mo?nost je trochu pracn?j?? a spo??v? ve vytvo?en? tabulky postupn? - za?neme se sloupcem `X_test` a postupn? p?id?me zb?vaj?c? dva.

In [ ]:
df_vysledek = pandas.DataFrame(X_test)
df_vysledek = df_vysledek.rename(columns={"review": "text"})
df_vysledek["y_test"] = y_test
df_vysledek["y_pred"] = y_pred
df_vysledek

Vytvo?en? tabulky se ??dky, kde se n?? model m?lil, je ?pln? stejn?.

In [ ]:
df_vysledek_incorrect = df_vysledek[df_vysledek["y_test"] != df_vysledek["y_pred"]]
df_vysledek_incorrect.to_csv("errors.csv", index=False)
df_vysledek_incorrect = df_vysledek_incorrect.reset_index()
df_vysledek_incorrect.head()

Pod?vejme se nyn? na p??klad recenze, kter? byla ohodnocen? ?patn?. Recenze se ve v?sledn?m souboru nach?z? na ??dku ?. 3 (p?i po??t?n? od 0). Text je n??e. Z textu je z?ejmn?, ?e autor nebo autorka se sna?il(a) napsat recenzi velmi zdvo?ile, co? asi n?? model zm?tlo.

*this was required reading for my british literature class . it 's the only novel in the class that i struggled with and only struggled from boredom . it is entirely possible that i just do not understand the genius that is forster , but i did not enjoy this novel . a lover of forster or modernist writing might enjoy it more . it must be a classic for a reason .*

Pod?vejme se ale na data d?kladn?ji. Vytvo?me si pandas tabulku s hodnotami koeficient?.

In [ ]:
df_coef = pandas.DataFrame(clf.coef_.T, columns=["coef"])
df_coef["words"] = vec.get_feature_names_out()
df_coef

D?le si vytvo??me pole pouze s textem t?to recenze a pomoc? `TfidfVectorizer` z n?j ud?l?me ??seln? pole. To p?evedeme na pandas tabulku.

In [ ]:
test_text = df_vysledek_incorrect.iloc[[3]]["text"]
test_text = vec.transform(test_text)
test_text = test_text.toarray()
test_text = pandas.DataFrame(test_text.T, columns=["coef"])
test_text = test_text[test_text["coef"] != 0]
test_text

Indexy v obou tabulk?ch odpov?daj? stejn?m slov?m nebo kombinac?m slov, m??eme tedy ob? tabulky propojit s vyu?it?m index?.

Sloupec `coef_x` ozna?uje hodnotu koeficientu, kter? slovu nebo kombinaci slov dal algoritmus SVM. Kladn? ??sla jsou sp??e ve prosp?ch hodnocen? `pos` a z?porn? ??sla ve prosp?ch hodnocen? `neg`. Sloupec `coef_y` ozna?uje koeficient, kter? slovu dal `TfidfVectorizer`.

In [ ]:
df_coef_merged = pandas.merge(df_coef, test_text, left_index=True, right_index=True)
df_coef_merged = df_coef_merged.sort_values(by=["coef_x"])
df_coef_merged

D?le si m??eme prov??it i n?jak? konkr?tn? slovo nebo kombinaci slov. Kup??kladu kombinace "required reading" m??e b?t lehce podez?el?, proto?e povinn? ?etba ?asto nepat?? mezi nejobl?ben?j?? kratochv?le studuj?c? ml?de?e. Pod?v?me se, kolik text? toto souslov? obsahuje a jak? jsou jejich skute?n? labels. Vych?zet budeme nyn? z tr?novac?ch dat, proto?e na nich se SVM "u?il" hodnoty koeficient?.

In [ ]:
word = "required reading"
df_selected_word = pandas.DataFrame({"text": X_train_text, "label": y_train})
df_selected_word = df_selected_word.dropna()
df_selected_word = df_selected_word[df_selected_word["text"].str.contains(word)]
df_selected_word.to_csv("df_selected_word.csv", index=False)
df_selected_word

Vid?me, ?e 3 ze 4 recenz? jsou pozitivn?, tak?e algoritmus pravd?podobn? postupoval spr?vn?.

**Bonus 2:** V?imni si, ?e soubor obsahuje tak? sloupec `product`, kter? identifikuje kategorii produktu.

1. Mohlo by b?t zaj?mav? se pod?vat, jestli je mo?n? data do t?chto kategori? shlukovat bez vyu?it? t?to informace. Data normalizuj (nap??klad pomoc? `TfidfVectorizer`), a zkus spustit n?jak? shlukovac? algoritmus, nap??klad `KMeans`. V?sledky shlukov?n? m??e? "obarvit" spr?vn?mi kategoriemi. Povedlo se modelu predikovat kategorie? 

2. Vyzkou?ej tak? klasifikaci do t?chto kategori?. Kter? kategorie se t??ko odli?uj??

Nejprve vyzkou??me vytvo?it shluky bez redukce dimenzionality. Vytvo?en? shluk? bez redukce dimenzionality je mo?n?, pouze jej nejde vizualizovat. M??eme ale vytvo?it graf metriky `inertia` pro r?zn? po?ty shluk?.

Proto?e s dvojicemi slov by v?po?ty b??ely opravdu dlouho, budeme uva?ovat pouze jednotliv? slova.

In [ ]:
X_1 = data["review"]
y = data["label"]

vec = TfidfVectorizer(stop_words="english")
X_1 = vec.fit_transform(X_1)
distances = []
cluster_counts = range(2, 15)

for k in cluster_counts:
    model = KMeans(n_clusters=k, n_init="auto", random_state=42)
    model.fit(X_1)
    distances.append(model.inertia_)

plt.plot(cluster_counts, distances,)

Nyn? vyzkou??me redukovat data do dvou rozm?r?, a to s vyu?it?m TSNE. Vid?me, ?e graf metriky `inertia` je "hlad??" a neust?le kles?.

Je t?eba uv?st, ?e TSNE nen? doporu?en? metoda pro redukci dimenzionality dat s velmi vysok?m po?tem sloupc?, co? na?e data ur?it? jsou. Proto pot? vyzkou??me je?t? algoritmus PCA (nap?. z d?vodu, ?e v?po?et trv? p??li? dlouho).

In [ ]:
X_2 = data["review"]
y = data["label"]

vec = TfidfVectorizer(stop_words="english")
X_2 = vec.fit_transform(X_2).toarray()

tsne = TSNE(
    n_components=2,
    random_state=42,
)
X_2 = tsne.fit_transform(X_2)

distances = []
cluster_counts = range(2, 15)

for k in cluster_counts:
    model = KMeans(n_clusters=k, n_init="auto", random_state=42)
    model.fit(X_2)
    distances.append(model.inertia_)

plt.plot(cluster_counts, distances,)

V?sledek m??eme vizualizovat s vyu?it?m dvourozm?rn?ho grafu. Vid?me, ?e z grafu bychom nepoznali, kolik shluk? je.

In [ ]:
plt.scatter(X_2[:, 0], X_2[:, 1], s=50)

P?id?me do grafu jednotliv? shluky a centroidy.

In [ ]:
model = KMeans(n_clusters=6, random_state=42, n_init="auto")
labels = model.fit_predict(X_2)
centers = model.cluster_centers_
plt.scatter(X_2[:, 0], X_2[:, 1], c=labels, s=50)
plt.scatter(centers[:, 0], centers[:, 1], c="black", s=200, alpha=0.5)

Nakonec vyzkou??me je?t? jin? zp?sob, jak sn??it dimenzionalitu. T?m je metoda PCA Principal Component Analysis (PCA), co? je jin? algoritmus pro redukci dimenzionality. Op?t redukujeme na 2 dimenze a vytvo??me si graf metriky `inertia`.

In [ ]:
X_3 = data["review"]
y = data["label"]

vec = TfidfVectorizer(stop_words="english")
X_3 = vec.fit_transform(X_3).toarray()

pca = PCA(
    n_components=2,
    random_state=42,
)
X_3 = pca.fit_transform(X_3)

distances = []
cluster_counts = range(2, 15)

for k in cluster_counts:
    model = KMeans(n_clusters=k, n_init="auto", random_state=42)
    model.fit(X_3)
    distances.append(model.inertia_)

plt.plot(cluster_counts, distances,)


Nyn? je samoz?ejm? ot?zka, kolik shluk? bychom se rozhodli na z?klad? grafu metriky vytvo?it. Osobn? bych volil 7, a to p?edev??m na z?klad? graf? s redukc? dimenzionality.

Pod?vejme se nyn?, kolik produkt? ve skute?nosti je. Je jich 6.

In [ ]:
data["product"].unique()

Vezm?me tedy p?vodn? data a pod?vejme se, jestli se li?? v?skyty produkt? v jednotliv?ch shluc?ch. Pou?ijeme 6 shluk?, tj. stejn? po?et shluk? jako produkt?. Porov?me v?sledky pro data v p?vodn? dimenzi a data s ob?ma postupy pro redukci dimenzionality.

In [ ]:
model = KMeans(n_clusters=6, random_state=42, n_init="auto")
labels_1 = model.fit_predict(X_1)
labels_2 = model.fit_predict(X_2)
labels_3 = model.fit_predict(X_3)
data["labels_1"] = labels_1
data["labels_2"] = labels_2
data["labels_3"] = labels_3

Porovn?n? provedeme pomoc? pivot tabulky. Jako ??dky zvol?me typy produkt? a jako sloupce zvol?me clustery. Vid?me, ?e po?ty v?skyt? jednotliv?ch typ? produkt? se v?razn? li??. Nap?. ve shluku 3 jsou t?m?? v?hradn? kamery, ve shluku 2 p?eva?uj? knihy, ve shluku 4 DVD atd.

Pokud bychom data ?etli po ??dc?ch, vid?me nap??klad, ?e drtiv? v?t?ina z hudebn?ch produkt? je ve shluku 1, tam je ale spousta dal??ch typ? produkt?.

In [ ]:
data.pivot_table(index="product", columns="labels_1", values="review", aggfunc=len, margins=True)

N??e jsou v?sledky pro redukci dimenzionality s algoritmem TSNE. Oproti p?edchoz?mu modelu jsou ale v?echny shluky p?ibli?n? stejn? po?etn?.

In [ ]:
data.pivot_table(index="product", columns="labels_2", values="review", aggfunc=len, margins=True)

Nakonec m?me v?sledky p?i redukci dimenzionality s vyu?it?m algoritmu PCA.

In [ ]:
data.pivot_table(index="product", columns="labels_3", values="review", aggfunc=len, margins=True)